# 决策树分类：像人类一样问问题做决策
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/分类 | 源文件：决策树_分类.py | 核心功能：树结构可视化、剪枝控制、GridSearchCV 超参数搜索

## 概述

决策树是最符合人类直觉的分类模型——它像一棵倒置的树，每个内部节点是一个"问题"（特征 <= 阈值？），每个分支是"答案"，每个叶节点是一个"决策"（类别预测）。

决策树最大的优势是**可解释性**：你可以直接把树的决策路径翻译成业务规则（"如果花瓣长度 <= 2.45cm，则是 Setosa"）。这在金融、医疗等需要解释模型决策的领域尤其重要。

脚本演示了决策树的训练、树结构文本展示、剪枝控制（max_depth），以及用 GridSearchCV 自动搜索最优超参数。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV

## 数学原理

### 1. 信息熵与不纯度

**代码对应**：`DecisionTreeClassifier(criterion="gini")` 和 `criterion="entropy"`。

**信息熵**（Entropy）：

$$H(S) = -\sum_{k=1}^{K} p_k \log_2 p_k$$

其中 $p_k$ 为集合 $S$ 中类别 $k$ 的比例。$H = 0$ 时最纯净（只有一类），$H = \log_2 K$ 时最混乱（各类等比例）。

**基尼系数**（Gini Impurity）：

$$\text{Gini}(S) = 1 - \sum_{k=1}^{K} p_k^2$$

基尼系数的直觉：从集合中随机抽取两个样本，它们类别**不同**的概率。$\text{Gini} = 0$ 时最纯净。

两者性质相近，实践中差异不大。sklearn 默认使用基尼系数（计算更快）。

### 2. 分裂准则：信息增益

**代码对应**：sklearn 内部在每个节点选择最优分裂。

对特征 $j$ 和阈值 $s$，分裂后的**信息增益**为：

$$\Delta H = H(S) - \left[\frac{|S_L|}{|S|}H(S_L) + \frac{|S_R|}{|S|}H(S_R)\right]$$

其中 $S_L = \{i: x_{ij} \leq s\}$，$S_R = \{i: x_{ij} > s\}$。

CART 算法遍历所有特征 $j$ 和所有可能阈值 $s$，选择使 $\Delta H$ 最大的 $(j, s)$ 组合。

### 3. 特征重要性

**代码对应**：`dt.feature_importances_` 返回特征重要性。

特征 $j$ 的重要性由其在所有节点中带来的不纯度减少量加权求和：

$$\text{Imp}(j) = \sum_{t: \text{splits on } j} \frac{n_t}{N}\Delta H_t$$

归一化后使所有特征重要性之和为 1。

### 4. 剪枝与正则化

**代码对应**：代码中 `for depth in [1, 2, 3, 5, 10, None]` 展示了深度控制的效果。

无限制的决策树会一直分裂直到每个叶节点只包含同一类样本（训练准确率 = 1.0），但这严重过拟合。

控制过拟合的超参数：
- `max_depth`：树的最大深度
- `min_samples_split`：节点分裂所需最少样本数
- `min_samples_leaf`：叶节点最少样本数

### 5. 预测概率

**代码对应**：`dt.predict_proba(X_test[:5])` 返回各类别的概率。

叶节点 $m$ 中类别 $k$ 的概率估计为：

$$\hat{P}(y=k|\mathbf{x} \in R_m) = \frac{|\{i \in R_m: y_i = k\}|}{|R_m|}$$

即叶节点中各类别样本的比例。如果叶节点只有 3 个样本，其中 2 个是类别 A，1 个是类别 B，则 $P(A) = 2/3$，$P(B) = 1/3$。

### 6. CART 算法的贪心策略

CART 使用贪心递归分裂：每次选择当前最优的 $(j, s)$，不考虑全局最优。这是 NP-hard 问题的近似解法。

**时间复杂度**：构建一棵深度为 $d$ 的平衡树，每层 $O(np\log n)$（排序），总复杂度 $O(npd\log n)$。

## 代码结构

| 段落 | 内容 |
|------|------|
| 数据 | Iris 数据集，80/20 分层分割 |
| 基础决策树 | 默认参数训练，展示深度、叶节点数、特征重要性 |
| 树结构展示 | export_text 以文本形式展示决策规则 |
| 剪枝对比 | max_depth 从 1 到无限制的过拟合曲线 |
| GridSearchCV | 搜索 criterion、max_depth、min_samples_split、min_samples_leaf |
| 概率预测 | 最优模型的概率输出和分类报告 |

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 2. 基础决策树

运行 2. 基础决策树 的代码，观察算法在该环节的行为。

In [ ]:
# criterion: "gini"(基尼系数) 或 "entropy"(信息增益)
dt = DecisionTreeClassifier(criterion="gini", max_depth=None, random_state=42)
dt.fit(X_train, y_train)

print("=== 决策树分类（默认参数）===")
print(f"训练集准确率: {dt.score(X_train, y_train):.4f}")
print(f"测试集准确率: {dt.score(X_test, y_test):.4f}")
print(f"树深度: {dt.get_depth()}")
print(f"叶节点数: {dt.get_n_leaves()}")
# --- 输出结果 ---
print(f"特征重要性: {dict(zip(iris.feature_names, dt.feature_importances_.round(4)))}")

### 3. 文本形式展示树结构

运行 3. 文本形式展示树结构 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 树结构（文本）===")
tree_rules = export_text(dt, feature_names=list(iris.feature_names), decimals=3)
print(tree_rules)

### 4. 剪枝控制复杂度

运行 4. 剪枝控制复杂度 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 不同 max_depth 对比 ===")
for depth in [1, 2, 3, 5, 10, None]:
    dt_d = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt_d.fit(X_train, y_train)
    train_acc = dt_d.score(X_train, y_train)
# --- 继续 ---
    test_acc = dt_d.score(X_test, y_test)
    depth_str = str(depth) if depth else "无限制"
    print(f"max_depth={depth_str:>3}: 训练={train_acc:.4f}, 测试={test_acc:.4f}, "
          f"叶节点={dt_d.get_n_leaves()}")

### 5. 超参数搜索

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== GridSearchCV 超参数搜索 ===")
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [2, 3, 4, 5, 6, 8, 10],
    "min_samples_split": [2, 5, 10],
# --- 继续 ---
    "min_samples_leaf": [1, 2, 4],
}
gs = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid, cv=5, scoring="accuracy", n_jobs=-1
# --- 继续 ---
)
gs.fit(X_train, y_train)
print(f"最佳参数: {gs.best_params_}")
print(f"最佳交叉验证准确率: {gs.best_score_:.4f}")
print(f"测试集准确率: {gs.best_estimator_.score(X_test, y_test):.4f}")

### 6. 预测概率

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print("\n=== 预测概率示例 ===")
dt_best = gs.best_estimator_
y_prob = dt_best.predict_proba(X_test[:5])
y_pred = dt_best.predict(X_test[:5])
for i in range(5):
# --- 循环处理 ---
    probs = ", ".join(f"{p:.3f}" for p in y_prob[i])
    print(f"样本{i+1}: 预测={iris.target_names[y_pred[i]]}, 概率=[{probs}], "
          f"真实={iris.target_names[y_test[i]]}")

y_pred_all = dt_best.predict(X_test)
print(f"\n分类报告:\n{classification_report(y_test, y_pred_all, target_names=iris.target_names)}")

## 关键代码解释

### 两种分裂准则

```python
dt = DecisionTreeClassifier(criterion="gini", max_depth=None, random_state=42)
```

- **gini**（基尼不纯度）：1 - Σp²，度量一个节点中样本的"混乱程度"
- **entropy**（信息增益）：-Σp·log₂p，基于信息论

实际效果差异很小，gini 计算更快（不需要对数运算），是默认选择。

### 树结构文本展示

```python
tree_rules = export_text(dt, feature_names=list(iris.feature_names), decimals=3)
```

export_text 把决策树导出为可读的文本规则。在不支持可视化的环境中（如服务器日志），这是理解模型决策逻辑的重要手段。

### 剪枝——防止过拟合

```python
for depth in [1, 2, 3, 5, 10, None]:
    dt_d = DecisionTreeClassifier(max_depth=depth, random_state=42)
```

不加限制的决策树会**完美记忆**训练数据（训练准确率 100%），但泛化能力极差。关键剪枝参数：
- max_depth：树的最大深度，最常用的剪枝手段
- min_samples_split：内部节点最少样本数
- min_samples_leaf：叶节点最少样本数

### GridSearchCV 自动调参

```python
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [2, 3, 4, 5, 6, 8, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring="accuracy", n_jobs=-1)
```

网格搜索 × 5 折交叉验证 = 暴力但有效的超参数搜索。
_jobs=-1 使用所有 CPU 核心并行计算。

## 使用示例

```python
from sklearn.tree import DecisionTreeClassifier, export_text

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=["特征1", "特征2"]))
print(dict(zip(feature_names, dt.feature_importances_)))
```

## 注意事项

1. **容易过拟合**：不限制深度的决策树几乎总是过拟合，必须做剪枝
2. **不稳定性**：数据的微小变化可能导致完全不同的树结构（集成方法如随机森林可以缓解）
3. **对类别不平衡敏感**：多数类主导分裂决策，考虑 class_weight="balanced"
4. **不需要特征缩放**：决策树基于阈值分裂，不受特征尺度影响
5. **特征重要性**：
eature_importances_ 基于不纯度降低量，但对高基数特征有偏（高基数特征更容易被选中）

## 总结与延伸

以上代码展示了 **决策树 分类** 的完整流程。

**解读要点**：
- 关注 **准确率（Accuracy）** 和 **分类报告** 中的精确率/召回率/F1
- 如果训练准确率远高于测试准确率，说明存在过拟合
- 观察 **混淆矩阵**，找出模型最容易混淆的类别

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **CART vs ID3 vs C4.5**：sklearn 使用 CART 算法（二叉树），ID3/C4.5 使用多叉树
- **决策树的方差**：单棵树方差大，bagging（随机森林）和 boosting（XGBoost）都是基于决策树的集成方法
- **SHAP 值**：比 
eature_importances_ 更可靠的特征重要性度量，能解释单个预测
- **多输出决策树**：DecisionTreeClassifier 支持多标签分类，每个叶节点输出多个类别
